# 03 — Tutor-response evidence notebook

This notebook turns parsed results into a structured evidence base for replying to the tutor. The central point is that the project is not trying to prove that LLMs possess embodied cognition. It tests whether image-schema prompting is a useful intermediate representation for controlled analysis of literal and metaphorical spatial language.

In [ ]:
from __future__ import annotations
import json
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
PARSED_PATH = DATA_DIR / "outputs" / "parsed_responses.jsonl"
RAW_PATH = DATA_DIR / "outputs" / "raw_responses.jsonl"
GOLD_PATH = DATA_DIR / "gold" / "sentences_v1.jsonl"

def read_jsonl(path: Path) -> pd.DataFrame:
    records = []
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if line.strip():
                records.append(json.loads(line))
    return pd.DataFrame(records)

def safe_accuracy(df: pd.DataFrame, pred_col: str, gold_col: str):
    if pred_col not in df.columns or gold_col not in df.columns:
        return None
    sub = df[df[pred_col].notna() & df[gold_col].notna()]
    if sub.empty:
        return None
    return float((sub[pred_col].astype(str) == sub[gold_col].astype(str)).mean())

def pct(x):
    return "NA" if x is None or pd.isna(x) else f"{100*x:.1f}%"

In [ ]:
parsed = read_jsonl(PARSED_PATH)
structured = parsed[parsed["parse_status"].eq("parsed")].copy()
print(f"Total parsed response records: {len(parsed)}")
print(f"Structured records available for accuracy: {len(structured)}")

In [ ]:
if structured.empty:
    print("No structured JSON records yet. Run direct-schema or structured-role jobs first.")
else:
    evidence = (
        structured.groupby(["prompt_family", "sentence_type"])
        .apply(lambda g: pd.Series({
            "n": len(g),
            "schema_accuracy": safe_accuracy(g, "main_image_schema", "expected_schema_primary"),
            "literal_metaphorical_accuracy": safe_accuracy(g, "literal_or_metaphorical", "expected_literal_or_metaphorical")
        }))
        .reset_index()
    )
    display(evidence)

In [ ]:
scenario_rows = [
    {"finding_pattern":"Literal and metaphorical accuracy are both high and similar",
     "interpretation":"Models can apply prompted image-schema categories across both uses in controlled short sentences.",
     "caution":"This does not prove embodied understanding; it may reflect linguistic regularities and instruction following.",
     "next_test":"Test less conventional examples and role/domain quality."},
    {"finding_pattern":"Literal accuracy is higher than metaphorical accuracy",
     "interpretation":"Models recognise surface spatial structure more reliably than abstract conceptual mappings.",
     "caution":"Could reflect ambiguity in metaphor gold labels or insufficient prompt guidance.",
     "next_test":"Analyse source_domain and target_domain errors separately."},
    {"finding_pattern":"Metaphorical accuracy is higher than literal accuracy",
     "interpretation":"Models may have learned image-schema terms from metaphor discourse and may over-apply them.",
     "caution":"Could be prompt-induced over-metaphorisation.",
     "next_test":"Check false positives on literal and control sentences."},
    {"finding_pattern":"Structured role prompt beats direct-schema prompt",
     "interpretation":"Explicit role decomposition improves consistency and interpretability.",
     "caution":"Better format compliance is not identical to deeper semantic understanding.",
     "next_test":"Evaluate role completeness and domain agreement separately."},
    {"finding_pattern":"Naïve prompt gives good ordinary interpretations",
     "interpretation":"Ordinary semantic interpretation and explicit image-schema analysis are separable outcomes.",
     "caution":"The project must show what image-schema prompting adds beyond paraphrase.",
     "next_test":"Compare meaning adequacy, schema accuracy, and role/domain explicitness."},
]
scenario_df = pd.DataFrame(scenario_rows)
display(scenario_df)

out = PROJECT_ROOT / "data" / "outputs" / "tutor_response_interpretation_matrix.csv"
out.parent.mkdir(parents=True, exist_ok=True)
scenario_df.to_csv(out, index=False)
print(f"Wrote: {out}")

## Draft reply position

A concise defensible position is:

> The project is not claiming that a correct image-schema label proves embodied cognition in an LLM. Instead, it asks whether image-schema prompting provides a useful intermediate representational layer for analysing literal and metaphorical spatial language. The key comparison is not merely whether the model can paraphrase the sentence correctly, but whether explicit schema and role prompting improves schema identification, literal/metaphorical discrimination, role completeness, and source–target domain recovery across sentence types.

This responds to the point that a naïve answer can be semantically good without mentioning metaphor. That becomes part of the experiment: ordinary meaning interpretation and explicit image-schema analysis are measured separately.

## Hypotheses

H1. Structured role-based prompting will produce higher primary-schema accuracy than direct-schema prompting.

H2. Literal spatial examples will be easier than metaphorical examples if models rely mainly on surface spatial cues.

H3. Metaphorical examples may be easier where expressions are strongly conventionalised in metaphor discourse.

H4. CONTAINER, SOURCE_PATH_GOAL, and VERTICALITY are expected to be more robust than SUPPORT_BALANCE because they are more visible in metaphor literature and common spatial vocabulary.

H5. A model can give a correct paraphrase without producing a correct image-schema analysis; this difference is central rather than incidental.